# 03 - Exploratory Data Analysis

This notebook examines the spatial and statistical relationships between modelled PM2.5, traffic exposure and green cover across London.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.config import PROCESSED_DIR

df = pd.read_parquet(PROCESSED_DIR / "london_pm25_final.parquet")
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

analysis_columns = [
    "pm25_mean", "major_mean_aadf", "major_hgv_mean_aadf",
    "major_road_length_m", "minor_vkm_total", "minor_hgv_vkm", "green_cover_pct",
]
summary = df[analysis_columns].describe().T
display(summary)

Rows: 6670
Columns: 10


,grid_id,pm25_mean,major_mean_aadf,major_hgv_mean_aadf,major_road_length_m,minor_vkm_total,minor_hgv_vkm,green_cover_pct,centroid_x,centroid_y
0,G00001,8.883902,24349.425376,2454.741385,2202.274839,2.011335e+05,9712.531133,16.734345,503750.0,175250.0
1,G00002,9.139251,37358.573847,3670.796720,2390.333577,1.783822e+05,8556.092427,23.696988,503750.0,175750.0
2,G00003,9.331372,49492.199098,4867.351405,1654.278829,1.218431e+07,521618.113937,2.297890,503750.0,176250.0
3,G00004,8.717595,91104.000000,9539.000000,119.478409,2.305859e+06,101952.088150,0.427522,503750.0,176750.0
4,G00005,7.481036,0.000000,0.000000,0.000000,3.446926e+05,16838.967983,2.491012,503750.0,189750.0


,count,mean,std,min,25%,50%,75%,max
pm25_mean,6670.0,8.773117,0.728961,7.033696,8.255383,8.738196,9.248711e+00,1.167439e+01
major_mean_aadf,6670.0,10408.604820,10267.447242,0.000000,2445.376038,9051.077821,1.481877e+04,9.143453e+04
major_hgv_mean_aadf,6670.0,452.345850,1142.727411,0.000000,5.212792,162.477278,4.821269e+02,1.401067e+04
major_road_length_m,6670.0,715.971566,671.742424,0.000000,128.459934,582.483829,1.058974e+03,5.364805e+03
minor_vkm_total,6670.0,965975.151272,995956.414354,0.000000,357942.667888,776735.121482,1.357797e+06,2.626114e+07
minor_hgv_vkm,6670.0,36447.941001,40013.120046,0.000000,12380.373456,28727.408321,5.244846e+04,9.975388e+05
green_cover_pct,6670.0,48.008400,26.467853,0.000000,27.442956,42.723161,6.743112e+01,1.000000e+02


In [2]:
correlation = df[analysis_columns].corr()
display(correlation.round(4))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
plots = [
    ("pm25_mean", "Annual mean PM2.5 (µg/m³)", "PM2.5 distribution", "hist"),
    ("green_cover_pct", "Green cover (%)", "Green-cover distribution", "hist"),
    ("green_cover_pct", "Green cover (%)", "PM2.5 and Green Cover", "scatter"),
    ("major_mean_aadf", "Mean major-road AADF", "PM2.5 and Major-Road Traffic", "scatter"),
    ("minor_vkm_total", "Minor-road vehicle-kilometres", "PM2.5 and Minor-Road Activity", "scatter"),
]
for ax, (column, xlabel, title, kind) in zip(axes.flat, plots):
    if kind == "hist":
        ax.hist(df[column], bins=40)
        ax.set_ylabel("Number of 500 m grid cells")
    else:
        ax.scatter(df[column], df["pm25_mean"], s=8, alpha=0.35)
        ax.set_ylabel("Annual mean PM2.5 (µg/m³)")
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

,pm25_mean,major_mean_aadf,major_hgv_mean_aadf,major_road_length_m,minor_vkm_total,minor_hgv_vkm,green_cover_pct
pm25_mean,1.0000,0.2905,0.0870,0.6196,0.4303,0.4248,-0.4918
major_mean_aadf,0.2905,1.0000,0.7675,0.3756,0.0929,0.0996,-0.1057
major_hgv_mean_aadf,0.0870,0.7675,1.0000,0.1896,-0.0090,0.0253,-0.0323
major_road_length_m,0.6196,0.3756,0.1896,1.0000,0.2348,0.2388,-0.3242
minor_vkm_total,0.4303,0.0929,-0.0090,0.2348,1.0000,0.9352,-0.3070
minor_hgv_vkm,0.4248,0.0996,0.0253,0.2388,0.9352,1.0000,-0.3188
green_cover_pct,-0.4918,-0.1057,-0.0323,-0.3242,-0.3070,-0.3188,1.0000


In [3]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(analysis_columns)))
ax.set_yticks(range(len(analysis_columns)))
ax.set_xticklabels(analysis_columns, rotation=45, ha="right")
ax.set_yticklabels(analysis_columns)
for i in range(len(analysis_columns)):
    for j in range(len(analysis_columns)):
        ax.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlation Matrix")
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

eda_summary = pd.DataFrame({
    "variable": analysis_columns,
    "mean": [df[c].mean() for c in analysis_columns],
    "median": [df[c].median() for c in analysis_columns],
    "std": [df[c].std() for c in analysis_columns],
    "minimum": [df[c].min() for c in analysis_columns],
    "maximum": [df[c].max() for c in analysis_columns],
})
display(eda_summary)

,variable,mean,median,std,minimum,maximum
0,pm25_mean,8.773117,8.738196,0.728961,7.033696,1.167439e+01
1,major_mean_aadf,10408.604820,9051.077821,10267.447242,0.000000,9.143453e+04
2,major_hgv_mean_aadf,452.345850,162.477278,1142.727411,0.000000,1.401067e+04
3,major_road_length_m,715.971566,582.483829,671.742424,0.000000,5.364805e+03
4,minor_vkm_total,965975.151272,776735.121482,995956.414354,0.000000,2.626114e+07
5,minor_hgv_vkm,36447.941001,28727.408321,40013.120046,0.000000,9.975388e+05
6,green_cover_pct,48.008400,42.723161,26.467853,0.000000,1.000000e+02
